In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [ ]:
import numpy as np
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, block_diag
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
from tqdm import tqdm
import pyreadr
from pathlib import Path

# ================================================================
# 0. CONFIG
# ================================================================

BASE_DIR = Path(r"D:\77\Research\temp\snow")
period = 52

burn = 1000
thin = 5
tot_save = 1000

a0 = 0.01
b0 = 0.01

eps_icar = 1e-6
eps_block = 1e-8

# 4-week pooling
Wk = 52
G = Wk // 4          # = 13 groups

# ================================================================
# 1. LOAD DATA
# ================================================================

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0]
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
print(f"S = {S}, T = {TT}")

# ================================================================
# 2. ADJACENCY + PROPER ICAR
# ================================================================

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
    crs="EPSG:4326"
)
gdf = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6

Dmat = squareform(pdist(xy))
W = (Dmat <= 0.22).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

deg = np.array(W.sum(axis=1)).flatten()
Q_icar = diags(deg) - W + eps_icar * diags(np.ones(S))
rank_icar = S

# ================================================================
# 3. TIME VARIABLES
# ================================================================

t_full = np.arange(1, TT + 1)
week_full = (t_full - 1) % 52
month_full = week_full // 4      # 0..12
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# 4. LOOP OVER p01 / p10
# ================================================================

for mode in ["p01", "p10"]:

    print(f"\n===== RUNNING {mode} (4-week τ, trend shared) =====")

    if mode == "p01":
        loc_mask = (y[:, :-1] == 0)
        kappa_func = lambda ny: ny - 0.5
    else:
        loc_mask = (y[:, :-1] == 1)
        kappa_func = lambda ny: (1 - ny) - 0.5

    loc = np.where(loc_mask)
    pairs = np.column_stack(loc)
    pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
    pairs[:, 1] += 1

    s_idx = pairs[:, 0]
    t_idx = pairs[:, 1] - 1
    N = len(s_idx)

    y_next = y[pairs[:, 0], pairs[:, 1]]
    kappa = kappa_func(y_next)

    t_raw = t_idx + 1
    week_idx = week_full[t_idx]
    month_idx = month_full[t_idx]
    t_trend = t_trend_full[t_idx]

    cov = np.column_stack([
        np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend
    ])

    K = 4
    theta_dim = K * Wk * S
    print(f"theta_dim = {theta_dim:,}")

    # ------------------------------------------------------------
    # DESIGN MATRIX (UNCHANGED)
    # ------------------------------------------------------------

    rows, cols, vals = [], [], []

    for i in tqdm(range(N), desc="Building X"):
        s = s_idx[i]
        w = week_idx[i]
        for k in range(K):
            b = k * Wk + w
            rows.append(i)
            cols.append(b * S + s)
            vals.append(cov[i, k])

    X = coo_matrix((vals, (rows, cols)),
                   shape=(N, theta_dim)).tocsr()

    # ------------------------------------------------------------
    # INITIAL VALUES
    # ------------------------------------------------------------

    theta = np.zeros(theta_dim)

    tau_month = np.ones((3, G))   # intercept / cos / sin
    tau_trend = 1.0               # shared

    all_theta = np.zeros((theta_dim, tot_save))
    all_tau_month = np.zeros((3, G, tot_save))
    all_tau_trend = np.zeros(tot_save)

    save_idx = 0
    total_iter = burn + tot_save * thin

    # ------------------------------------------------------------
    # MCMC
    # ------------------------------------------------------------

    for it in tqdm(range(total_iter), desc="MCMC"):

        phi = X @ theta
        omega = random_polyagamma(1, phi, size=N)

        # -----------------------------
        # PRIOR PRECISION
        # -----------------------------
        P_blocks = []

        for k in range(K):
            for w in range(Wk):

                if k < 3:
                    tau_eff = tau_month[k, w // 4]
                else:
                    tau_eff = tau_trend

                P_blocks.append(
                    tau_eff * Q_icar + eps_block * diags(np.ones(S))
                )

        P0 = block_diag(P_blocks, format="csr")

        XtOmega = X.T.multiply(omega)
        post_prec = XtOmega @ X + P0

        factor = cholesky(post_prec)
        mu = factor.solve_A(X.T @ kappa)
        theta = mu + factor.solve_A(np.random.randn(theta_dim))

        # -----------------------------
        # τ updates
        # -----------------------------

        # month-wise (intercept / cos / sin)
        for k in range(3):
            for m in range(G):
                quad = 0.0
                for w in range(m*4, min((m+1)*4, Wk)):
                    b = k * Wk + w
                    beta = theta[b*S:(b+1)*S]
                    quad += beta @ (Q_icar @ beta)

                tau_month[k, m] = np.random.gamma(
                    shape=a0 + 0.5 * rank_icar * 4,
                    scale=1.0 / (b0 + 0.5 * quad)
                )

        # shared trend
        quad_trend = 0.0
        for w in range(Wk):
            b = 3 * Wk + w
            beta = theta[b*S:(b+1)*S]
            quad_trend += beta @ (Q_icar @ beta)

        tau_trend = np.random.gamma(
            shape=a0 + 0.5 * rank_icar * Wk,
            scale=1.0 / (b0 + 0.5 * quad_trend)
        )

        # -----------------------------
        # SAVE
        # -----------------------------
        if it >= burn and (it - burn) % thin == 0:
            all_theta[:, save_idx] = theta
            all_tau_month[:, :, save_idx] = tau_month
            all_tau_trend[save_idx] = tau_trend
            save_idx += 1
            if save_idx == tot_save:
                break

    # ------------------------------------------------------------
    # SAVE RESULTS
    # ------------------------------------------------------------

    np.savez_compressed(
        BASE_DIR / f"icar_{mode}_4week_tau_partial.npz",
        all_theta=all_theta,
        all_tau_month=all_tau_month,
        all_tau_trend=all_tau_trend
    )

print("\n===== ALL DONE: 4-week τ + shared trend τ =====")


S = 1601, T = 2704

===== RUNNING p01 (4-week τ, trend shared) =====
theta_dim = 333,008


Building X:  99%|█████████▉| 2742564/2775364 [00:07<00:00, 353912.45it/s]